In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col

In [0]:
df = (
    spark.read
    .format("json")
    .option("inferSchema", True)
    .option("multiLine", True)
    .load("/Volumes/workspace/streaming/json_files/day1.json")
)

In [0]:
df.display()

customer,items,metadata,order_id,payment,timestamp
"List(List(Toronto, Canada, M5H 2N2), 501, john@example.com, John Doe)","List(List(I100, 25.99, Wireless Mouse, 2), List(I101, 15.49, USB-C Adapter, 1))","List(List(campaign, back_to_school), List(channel, email))",ORD1001,"List(Credit Card, TXN7890)",2025-06-01T10:15:00Z


### Flattening the columns

#### 1. customer column
##### Customer column contains a nested json

In [0]:
(
    df.select("order_id", "timestamp", "customer.address.city",
              "customer.address.country", "customer.address.postal_code",
              "customer.customer_id", "customer.email", "customer.name")
    .display()
 )

order_id,timestamp,city,country,postal_code,customer_id,email,name
ORD1001,2025-06-01T10:15:00Z,Toronto,Canada,M5H 2N2,501,john@example.com,John Doe


#### 2. items column
##### Items column is a  list of dictionaries
- So, first explode the lists into rows
- To do this, explode() or explode_outer() is used
- explode ignores the null whereas explode_outer() keeps the null
- For each list_element i.e dictionary, new row will be assigned. 
- So the number of rows will depend upon the number of dictionaries inside the list
- Columns without list values will have their existing values duplicated for each generated row.

In [0]:
item_df = df.withColumn("items", F.explode_outer(col("items")))

In [0]:
item_df.display()

customer,items,metadata,order_id,payment,timestamp
"List(List(Toronto, Canada, M5H 2N2), 501, john@example.com, John Doe)","List(I100, 25.99, Wireless Mouse, 2)","List(List(campaign, back_to_school), List(channel, email))",ORD1001,"List(Credit Card, TXN7890)",2025-06-01T10:15:00Z
"List(List(Toronto, Canada, M5H 2N2), 501, john@example.com, John Doe)","List(I101, 15.49, USB-C Adapter, 1)","List(List(campaign, back_to_school), List(channel, email))",ORD1001,"List(Credit Card, TXN7890)",2025-06-01T10:15:00Z


In [0]:
item_df.select("items.item_id", "items.price", "items.product_name", "items.quantity").display()

item_id,price,product_name,quantity
I100,25.99,Wireless Mouse,2
I101,15.49,USB-C Adapter,1


#### 3. metadata column
- This column also contains list of dictionaries

In [0]:
meta_df = df.withColumn("metadata", F.explode_outer(col("metadata")))

In [0]:
meta_df.display()

customer,items,metadata,order_id,payment,timestamp
"List(List(Toronto, Canada, M5H 2N2), 501, john@example.com, John Doe)","List(List(I100, 25.99, Wireless Mouse, 2), List(I101, 15.49, USB-C Adapter, 1))","List(campaign, back_to_school)",ORD1001,"List(Credit Card, TXN7890)",2025-06-01T10:15:00Z
"List(List(Toronto, Canada, M5H 2N2), 501, john@example.com, John Doe)","List(List(I100, 25.99, Wireless Mouse, 2), List(I101, 15.49, USB-C Adapter, 1))","List(channel, email)",ORD1001,"List(Credit Card, TXN7890)",2025-06-01T10:15:00Z


In [0]:
meta_df.select("metadata.key", "metadata.value").display()

key,value
campaign,back_to_school
channel,email


#### 4. payment column
- This column has a simple dictionary

In [0]:
df.select("payment.method", "payment.transaction_id").display()

method,transaction_id
Credit Card,TXN7890


### Now loading the files, transforming and storing them in delta tables
- There are three files inside the json_files volume: day1.json, day2.json and day3.json
- When reading data using the readStream method, always specify the path of the folder not the file
- In batch processing, specifying the file path is good but not in stream processing
- For writing stream data -> .toTable()
- For writing batch data -> .saveAsTable()

In [0]:
df_batch = (
        spark.read.format("json")
        .option("multiline", True)
        .load("/Volumes/workspace/streaming/json_files/day1.json")
    )

file_schema = df_batch.schema

df = (
            spark.readStream.format("json")
            .option("multiline", True)
            .schema(file_schema)
            .load("/Volumes/workspace/streaming/json_files/")
        )

df = (
        df.withColumn("items", F.explode_outer(col("items")))
        .withColumn("metadata", F.explode_outer(col("metadata")))
        )

df = df.select(
            "customer.customer_id",
            "order_id",
            "customer.email",
            "customer.name",
            "customer.address.city",
            "customer.address.country",
            "customer.address.postal_code",
            "payment.method",
            "payment.transaction_id",
            "timestamp",
            "items.item_id",
            "items.price",
            "items.product_name",
            "items.quantity",
            "metadata.key",
            "metadata.value"
        )

(
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/streaming/checkpoints/days")
    .trigger(once = True)
    .toTable("workspace.streaming.days")
)

In [0]:
df = spark.table("workspace.streaming.days")

In [0]:
df.display()

customer_id,order_id,email,name,city,country,postal_code,method,transaction_id,timestamp,item_id,price,product_name,quantity,key,value
503,ORD1003,david@example.com,David Lee,Calgary,Canada,T2P 1G1,Debit Card,TXN7892,2025-06-01T11:00:00Z,I103,199.99,Noise Cancelling Headphones,1,referrer,instagram
503,ORD1003,david@example.com,David Lee,Calgary,Canada,T2P 1G1,Debit Card,TXN7892,2025-06-01T11:00:00Z,I103,199.99,Noise Cancelling Headphones,1,coupon,WELCOME10
503,ORD1003,david@example.com,David Lee,Calgary,Canada,T2P 1G1,Debit Card,TXN7892,2025-06-01T11:00:00Z,I104,29.99,Laptop Stand,1,referrer,instagram
503,ORD1003,david@example.com,David Lee,Calgary,Canada,T2P 1G1,Debit Card,TXN7892,2025-06-01T11:00:00Z,I104,29.99,Laptop Stand,1,coupon,WELCOME10
503,ORD1003,david@example.com,David Lee,Calgary,Canada,T2P 1G1,Debit Card,TXN7892,2025-06-01T11:00:00Z,I105,10.0,HDMI Cable,2,referrer,instagram
503,ORD1003,david@example.com,David Lee,Calgary,Canada,T2P 1G1,Debit Card,TXN7892,2025-06-01T11:00:00Z,I105,10.0,HDMI Cable,2,coupon,WELCOME10
501,ORD1001,john@example.com,John Doe,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,2025-06-01T10:15:00Z,I100,25.99,Wireless Mouse,2,campaign,back_to_school
501,ORD1001,john@example.com,John Doe,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,2025-06-01T10:15:00Z,I100,25.99,Wireless Mouse,2,channel,email
501,ORD1001,john@example.com,John Doe,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,2025-06-01T10:15:00Z,I101,15.49,USB-C Adapter,1,campaign,back_to_school
501,ORD1001,john@example.com,John Doe,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,2025-06-01T10:15:00Z,I101,15.49,USB-C Adapter,1,channel,email
